In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import make_pipeline

from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score
)

import joblib

print("Libraries Imported Successfully!")

Libraries Imported Successfully!


In [11]:
df = pd.read_csv("cleaned_data.csv")

In [12]:
y_clf = (df["charges"] > df["charges"].median()).astype(int)

X = df.drop("charges", axis=1)

In [13]:
categorical_columns = X.select_dtypes(include=["object", "category"]).columns

print("Categorical Columns:")
print(categorical_columns)

X = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True
)

print("\nColumns After Encoding:")
print(X.columns)

Categorical Columns:
Index(['sex', 'smoker', 'region'], dtype='object')

Columns After Encoding:
Index(['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest',
       'region_southeast', 'region_southwest'],
      dtype='object')


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_clf,
    test_size=0.20,
    random_state=42,
    stratify=y_clf
)

In [15]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Scaling Completed Successfully!")

Scaling Completed Successfully!


In [16]:
# Task 1: Decision Tree Baseline

In [17]:
decision_tree = DecisionTreeClassifier(
    random_state=42
)

decision_tree.fit(
    X_train_scaled,
    y_train
)

print("Decision Tree Trained Successfully!")

Decision Tree Trained Successfully!


In [18]:
train_prediction = decision_tree.predict(X_train_scaled)

train_accuracy = accuracy_score(
    y_train,
    train_prediction
)

print("Training Accuracy:", train_accuracy)

Training Accuracy: 1.0


In [19]:
test_prediction = decision_tree.predict(X_test_scaled)

test_accuracy = accuracy_score(
    y_test,
    test_prediction
)

print("Testing Accuracy:", test_accuracy)

Testing Accuracy: 0.9067164179104478


In [20]:
comparison = pd.DataFrame({

    "Dataset":[
        "Training",
        "Testing"
    ],

    "Accuracy":[
        train_accuracy,
        test_accuracy
    ]

})

display(comparison)

,Dataset,Accuracy
0,Training,1.000000
1,Testing,0.906716


In [21]:
# Task 2: Controlled Decision Tree

In [22]:
controlled_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

controlled_tree.fit(
    X_train_scaled,
    y_train
)

print("Controlled Decision Tree Trained Successfully!")

Controlled Decision Tree Trained Successfully!


In [23]:
controlled_train_pred = controlled_tree.predict(X_train_scaled)

controlled_train_accuracy = accuracy_score(
    y_train,
    controlled_train_pred
)

print("Training Accuracy:", controlled_train_accuracy)

Training Accuracy: 0.9289055191768008


In [24]:
controlled_test_pred = controlled_tree.predict(X_test_scaled)

controlled_test_accuracy = accuracy_score(
    y_test,
    controlled_test_pred
)

print("Testing Accuracy:", controlled_test_accuracy)

Testing Accuracy: 0.9365671641791045


In [25]:
comparison = pd.DataFrame({

    "Model": [
        "Uncontrolled Tree",
        "Controlled Tree"
    ],

    "Training Accuracy": [
        train_accuracy,
        controlled_train_accuracy
    ],

    "Testing Accuracy": [
        test_accuracy,
        controlled_test_accuracy
    ]

})

display(comparison)

,Model,Training Accuracy,Testing Accuracy
0,Uncontrolled Tree,1.000000,0.906716
1,Controlled Tree,0.928906,0.936567


In [26]:
# Task 3: Gini vs Entropy Comparison

In [27]:
gini_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

gini_tree.fit(X_train_scaled, y_train)

gini_pred = gini_tree.predict(X_test_scaled)

gini_accuracy = accuracy_score(y_test, gini_pred)

print("Gini Test Accuracy:", gini_accuracy)

Gini Test Accuracy: 0.9402985074626866


In [28]:
entropy_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

entropy_tree.fit(X_train_scaled, y_train)

entropy_pred = entropy_tree.predict(X_test_scaled)

entropy_accuracy = accuracy_score(y_test, entropy_pred)

print("Entropy Test Accuracy:", entropy_accuracy)

Entropy Test Accuracy: 0.9253731343283582


In [29]:
comparison = pd.DataFrame({
    "Criterion": ["Gini", "Entropy"],
    "Test Accuracy": [gini_accuracy, entropy_accuracy]
})

display(comparison)

,Criterion,Test Accuracy
0,Gini,0.940299
1,Entropy,0.925373


In [30]:
# Task 4: Random Forest Classifier

In [31]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

random_forest.fit(X_train_scaled, y_train)

print("Random Forest Trained Successfully!")

Random Forest Trained Successfully!


In [32]:
rf_train_pred = random_forest.predict(X_train_scaled)
rf_test_pred = random_forest.predict(X_test_scaled)

rf_train_accuracy = accuracy_score(y_train, rf_train_pred)
rf_test_accuracy = accuracy_score(y_test, rf_test_pred)

print("Training Accuracy :", rf_train_accuracy)
print("Testing Accuracy  :", rf_test_accuracy)

Training Accuracy : 0.9635173058933583
Testing Accuracy  : 0.9477611940298507


In [33]:
rf_prob = random_forest.predict_proba(X_test_scaled)[:,1]

rf_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest ROC-AUC :", rf_auc)

Random Forest ROC-AUC : 0.9487079527734462


In [34]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": random_forest.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

display(importance.head(5))

,Feature,Importance
0,age,0.516106
4,smoker_yes,0.293830
1,bmi,0.110592
2,children,0.041152
3,sex_male,0.013884


In [35]:
# Task 4a: Gradient Boosting Classifier

In [36]:
from sklearn.ensemble import GradientBoostingClassifier

gradient_boost = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gradient_boost.fit(
    X_train_scaled,
    y_train
)

print("Gradient Boosting Model Trained Successfully!")

Gradient Boosting Model Trained Successfully!


In [37]:
gb_train_pred = gradient_boost.predict(X_train_scaled)
gb_test_pred = gradient_boost.predict(X_test_scaled)

gb_train_accuracy = accuracy_score(y_train, gb_train_pred)
gb_test_accuracy = accuracy_score(y_test, gb_test_pred)

print("Training Accuracy :", gb_train_accuracy)
print("Testing Accuracy :", gb_test_accuracy)

Training Accuracy : 0.9560336763330215
Testing Accuracy : 0.9328358208955224


In [38]:
gb_prob = gradient_boost.predict_proba(X_test_scaled)[:, 1]

gb_auc = roc_auc_score(y_test, gb_prob)

print("Gradient Boosting ROC-AUC :", gb_auc)

Gradient Boosting ROC-AUC : 0.9501837825796392


In [39]:
least_important = importance.sort_values(
    by="Importance",
    ascending=True
).head(5)

display(least_important)

,Feature,Importance
7,region_southwest,0.006957
5,region_northwest,0.007155
6,region_southeast,0.010325
3,sex_male,0.013884
2,children,0.041152


In [40]:
features_to_remove = least_important["Feature"].tolist()

print("Features Removed:")
print(features_to_remove)

X_reduced = X.drop(columns=features_to_remove)

Features Removed:
['region_southwest', 'region_northwest', 'region_southeast', 'sex_male', 'children']


In [41]:
X_train_red, X_test_red, y_train_red, y_test_red = train_test_split(
    X_reduced,
    y_clf,
    test_size=0.20,
    random_state=42,
    stratify=y_clf
)

scaler_red = StandardScaler()

X_train_red = scaler_red.fit_transform(X_train_red)
X_test_red = scaler_red.transform(X_test_red)

In [42]:
rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(X_train_red, y_train_red)

reduced_prob = rf_reduced.predict_proba(X_test_red)[:,1]

reduced_auc = roc_auc_score(
    y_test_red,
    reduced_prob
)

print("Original Random Forest AUC :", rf_auc)
print("Reduced Random Forest AUC :", reduced_auc)

Original Random Forest AUC : 0.9487079527734462
Reduced Random Forest AUC : 0.9396302071730898


In [43]:
comparison = pd.DataFrame({
    "Model": [
        "Full Random Forest",
        "Reduced Random Forest"
    ],
    "ROC-AUC": [
        rf_auc,
        reduced_auc
    ]
})

display(comparison)

,Model,ROC-AUC
0,Full Random Forest,0.948708
1,Reduced Random Forest,0.939630


In [44]:
# Task 5: 5-Fold Cross Validation

In [45]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Controlled Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=20,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y_clf,
        cv=cv,
        scoring="roc_auc"
    )

    results.append({
        "Model": name,
        "Mean AUC": scores.mean(),
        "Std AUC": scores.std()
    })

cv_results = pd.DataFrame(results)

display(cv_results)

,Model,Mean AUC,Std AUC
0,Logistic Regression,0.947334,0.014940
1,Controlled Decision Tree,0.934183,0.011578
2,Random Forest,0.949528,0.013619
3,Gradient Boosting,0.950935,0.010529


In [46]:
# Task 6: GridSearchCV with Pipeline

In [47]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV

pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

In [49]:
param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5]
}

In [51]:
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('simpleimputer',
                                        SimpleImputer(strategy='median')),
                                       ('standardscaler', StandardScaler()),
                                       ('randomforestclassifier',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'randomforestclassifier__max_depth': [5, 10, None],
                         'randomforestclassifier__min_samples_leaf': [1, 5],
                         'randomforestclassifier__n_estimators': [50, 100,
                                                                  200]},
             scoring='roc_auc')

In [52]:
print("Best Parameters:")
print(grid.best_params_)

print()

print("Best Cross-Validation AUC:")
print(grid.best_score_)

Best Parameters:
{'randomforestclassifier__max_depth': 10, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 200}

Best Cross-Validation AUC:
0.9553059754744998


In [53]:
total_models = (
    len(param_grid["randomforestclassifier__n_estimators"])
    * len(param_grid["randomforestclassifier__max_depth"])
    * len(param_grid["randomforestclassifier__min_samples_leaf"])
)

print("Total Parameter Combinations:", total_models)
print("Total Models Evaluated:", total_models * 5)

Total Parameter Combinations: 18
Total Models Evaluated: 90


In [54]:
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

learning_curve_results = []

best_pipeline = grid.best_estimator_

for fraction in fractions:

    size = int(fraction * len(X_train))

    X_subset = X_train.iloc[:size]
    y_subset = y_train.iloc[:size]

    best_pipeline.fit(X_subset, y_subset)

    # Training AUC
    train_prob = best_pipeline.predict_proba(X_subset)[:, 1]

    train_auc = roc_auc_score(
        y_subset,
        train_prob
    )

    # Test AUC
    test_prob = best_pipeline.predict_proba(X_test)[:, 1]

    test_auc = roc_auc_score(
        y_test,
        test_prob
    )

    learning_curve_results.append([
        fraction,
        train_auc,
        test_auc
    ])

learning_curve_df = pd.DataFrame(
    learning_curve_results,
    columns=[
        "Training Fraction",
        "Training AUC",
        "Test AUC"
    ]
)

display(learning_curve_df)

,Training Fraction,Training AUC,Test AUC
0,0.2,1.000000,0.959067
1,0.4,1.000000,0.949237
2,0.6,0.999971,0.946425
3,0.8,0.999995,0.950601
4,1.0,0.999748,0.951493


In [55]:
joblib.dump(best_pipeline, "best_model.pkl")

['best_model.pkl']

In [56]:
joblib.dump(
    best_pipeline,
    "best_model.pkl"
)

print("Model Saved Successfully!")

Model Saved Successfully!


In [57]:
loaded_model = joblib.load("best_model.pkl")

sample_rows = X.iloc[:2]

predictions = loaded_model.predict(sample_rows)

print("Predictions:")

print(predictions)

Predictions:
[1 0]


In [58]:
from google.colab import files

files.download("best_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:
fractions=[0.2,0.4,0.6,0.8,1.0]
results=[]
best_pipeline=grid.best_estimator_

for f in fractions:
    size=int(f*len(X_train))
    X_sub=X_train.iloc[:size]
    y_sub=y_train.iloc[:size]

    best_pipeline.fit(X_sub,y_sub)

    train_auc=roc_auc_score(y_sub,best_pipeline.predict_proba(X_sub)[:,1])
    test_auc=roc_auc_score(y_test,best_pipeline.predict_proba(X_test)[:,1])

    results.append([f,train_auc,test_auc])

learning_curve_df=pd.DataFrame(results,
columns=["Training Fraction","Training AUC","Test AUC"])

display(learning_curve_df)

,Training Fraction,Training AUC,Test AUC
0,0.2,1.000000,0.959067
1,0.4,1.000000,0.949237
2,0.6,0.999971,0.946425
3,0.8,0.999995,0.950601
4,1.0,0.999748,0.951493


In [60]:
joblib.dump(grid.best_estimator_,"best_model.pkl")
print("Saved!")

Saved!


In [61]:
loaded_model=joblib.load("best_model.pkl")

print(loaded_model.predict(X.iloc[:2]))

[1 0]
